# Finetuning GPT-2 large

GPT-2 (large):  

- 774 million parameters
- 36 layers
- Hidden size: 1280
- 20 attention heads

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import pipeline

C:\Users\micha\anaconda3\envs\transformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset('vicclab/fairy_tales')

In [3]:
train_val = dataset["train"].train_test_split(
    test_size=0.2, seed=42)

In [4]:
dataset = DatasetDict({
    "train": train_val["train"],
    "validation": train_val["test"]
})

In [5]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Train size: 82878
Validation size: 20720


In [6]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2-large')
tokenizer.pad_token = tokenizer.eos_token

In [7]:
def tokenize_function(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )
    return enc

In [8]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map: 100%|█████████████████████████████████████████████████████████████| 20720/20720 [00:00<00:00, 36967.44 examples/s]


In [9]:
tokenized_datasets = tokenized_datasets.filter(
    lambda x: len(x["input_ids"]) > 0
)

Filter: 100%|██████████████████████████████████████████████████████████| 20720/20720 [00:00<00:00, 60026.23 examples/s]


In [10]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2-large"
).to("cuda")

training_args = TrainingArguments(
    output_dir="models/results",
    eval_strategy="epoch",
    num_train_epochs=5,             
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="models/logs",
    save_strategy="epoch",
    report_to="none"
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.703000,4.021769
2,1.329300,4.187929
3,1.056100,4.400712
4,0.680700,4.712324
5,0.495800,4.928694


TrainOutput(global_step=83275, training_loss=1.0466716393247735, metrics={'train_runtime': 6081.0972, 'train_samples_per_second': 54.775, 'train_steps_per_second': 13.694, 'total_flos': 2.8100373715968e+16, 'train_loss': 1.0466716393247735, 'epoch': 5.0})

---